# LoRA Fine-Tuning (Colab)

## 1. Install Dependencies

In [ ]:
!pip install -q torch torchvision transformers peft accelerate trl datasets wandb sentencepiece protobuf safetensors

## 2. Mount Google Drive & Login (W&B, HuggingFace)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import wandb
wandb.login()

In [ ]:
from huggingface_hub import login
login()

## 3. Configuration

Adjust paths and hyperparameters here. Output goes to Google Drive so it persists after runtime disconnects.

In [ ]:
# --- Model & Data ---
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
# MODEL_ID = "Qwen/Qwen3-8B"

# Path to dataset (upload to Drive or Colab files)
DATASET_PATH = "/content/drive/MyDrive/patent_datasets/task_1_multilingual.json"

# Output directory on Google Drive (persists after disconnect)
OUTPUT_DIR = "/content/drive/MyDrive/lora_output/task_llama"

# --- Sequence & Training ---
MAX_SEQ_LENGTH = 8192
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4  # effective batch = 2 * 4 = 8
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.05
EVAL_RATIO = 0.10
LR_SCHEDULER_TYPE = "cosine"
SEED = 42

# --- LoRA Hyper-parameters ---
LORA_R = 16
LORA_ALPHA = 32
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

# --- Precision & Memory ---
USE_BF16 = True
GRADIENT_CHECKPOINTING = True

# --- Logging & Saving ---
LOGGING_STEPS = 5
EVAL_AND_SAVE_STEP = 25
SAVE_TOTAL_LIMIT = 25
REPORT_TO = "wandb"

## 4. Load Dataset

In [ ]:
import json
from datasets import Dataset


def load_patent_dataset(path: str, eval_ratio: float = 0.1):
    with open(path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    formatted = []
    for entry in raw_data:
        if "mistral" in MODEL_ID.lower():
            messages = [
                {"role": "user", "content": entry["instruction"] + "\n\n" + entry["input"]},
                {"role": "assistant", "content": entry["output"]},
            ]
        else:
            messages = [
                {"role": "system", "content": entry["instruction"]},
                {"role": "user", "content": entry["input"]},
                {"role": "assistant", "content": entry["output"]},
            ]
        formatted.append({"messages": messages})

    dataset = Dataset.from_list(formatted)

    if eval_ratio > 0:
        split = dataset.train_test_split(test_size=eval_ratio, seed=SEED)
        return split["train"], split["test"]
    return dataset, None


train_dataset, eval_dataset = load_patent_dataset(DATASET_PATH, eval_ratio=EVAL_RATIO)
print(f"Train samples: {len(train_dataset)}")
if eval_dataset:
    print(f"Eval samples:  {len(eval_dataset)}")

## 5. Load Model & Tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Enable TF32 for faster matmul on A100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

attn_impl = "sdpa"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    attn_implementation=attn_impl,
    device_map="auto",
)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
print(f"Model loaded (bf16 + {attn_impl}, no quantization).")

## 6. Configure LoRA & Train

In [ ]:
import os
from peft import LoraConfig, TaskType
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_SEQ_LENGTH,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    bf16=USE_BF16,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=LOGGING_STEPS,
    save_steps=EVAL_AND_SAVE_STEP,
    save_total_limit=SAVE_TOTAL_LIMIT,
    save_strategy="steps",
    eval_strategy="steps",
    eval_steps=EVAL_AND_SAVE_STEP,
    per_device_eval_batch_size=1,
    seed=SEED,
    report_to=REPORT_TO,
    torch_compile=True,
    torch_compile_backend="inductor",
    load_best_model_at_end=True,
    optim="adamw_torch_fused",
    tf32=True,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()

## 7. Save Final Adapter to Drive

In [ ]:
final_dir = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"LoRA adapter saved to: {final_dir}")

wandb.finish()
print("W&B run finished.")